###Import libraries

In [ ]:
!pip install transformers datasets scikit-learn torch -q

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

###Load and prepare dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ButterChicken98/plantvillage-image-text-pairs",
    split="train"
)

df = dataset.to_pandas()
df.head()

In [ ]:
df.rename(columns={'caption': 'class_name'}, inplace=True)

In [ ]:
df['text'] = df['captions'].apply(lambda x: x[0])

In [ ]:
df = df[['text', 'class_name']]
df.head()

### Encode labels and split data

In [ ]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['class_name'])

num_labels = len(label_encoder.classes_)
print("Number of classes:", num_labels)

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

###Tokenize

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [ ]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask'],
    'labels': val_labels.tolist()
})

###Define model and training arguments

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # NEW correct keyword
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

Train model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

###Evaluate model

In [ ]:
predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(val_labels, preds))
print(classification_report(val_labels, preds))

In [ ]:
def predict_disease(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([predicted_class])[0]

In [ ]:
predict_disease("Tomato leaf showing yellow spots and fungal infection")

###Save model

In [ ]:
# Save model
model.save_pretrained("./plant_disease_text_model")

# Save tokenizer
tokenizer.save_pretrained("./plant_disease_text_model")

# Save label encoder
import pickle

with open("./plant_disease_text_model/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

In [ ]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model = DistilBertForSequenceClassification.from_pretrained("./plant_disease_text_model")
tokenizer = DistilBertTokenizerFast.from_pretrained("./plant_disease_text_model")